# Notebook 11 — Handling Imbalanced Data
### Sprint 5 | Data Cleaning & Preprocessing for AI/ML Engineers


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

df = pd.read_csv("telco_churn.csv")
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

encode_cols = df.select_dtypes(include='object').columns.drop(['customerID', 'Churn'])
df_encoded = df.copy()
for col in encode_cols:
    df_encoded[col] = LabelEncoder().fit_transform(df_encoded[col])

X = df_encoded.drop(columns=['customerID', 'Churn'])
y = (df['Churn'] == 'Yes').astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f"Train: {len(X_train):,} rows | Test: {len(X_test):,} rows")


Train: 5,634 rows | Test: 1,409 rows


/tmp/ipykernel_783/1745770289.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  encode_cols = df.select_dtypes(include='object').columns.drop(['customerID', 'Churn'])


---
## 1. What is Class Imbalance? & 2. Balanced vs Imbalanced Dataset

### Understand
A dataset is imbalanced when one class substantially outnumbers another. There's no
single hard cutoff, but a common rule of thumb: under ~1.5:1 is roughly balanced, 1.5:1
to 10:1 is moderate imbalance, beyond 10:1 (or 100:1+ in fraud/rare-disease contexts) is
severe.

### Demonstrate
**This dataset's exact numbers (Sprint 4, Notebook 10):** `Churn` splits 73.46% No /
26.54% Yes — a 2.77:1 ratio. **Moderate**, not severe.


In [2]:
print(y.value_counts())
print((y.value_counts(normalize=True)*100).round(2))


Churn
0    5174
1    1869
Name: count, dtype: int64
Churn
0    73.46
1    26.54
Name: proportion, dtype: float64


---
## 3. Why Class Imbalance is a Problem

### Understand
Most algorithms implicitly optimize for overall accuracy — with an imbalanced target,
the easiest way to raise accuracy is to just favor the majority class, since it dominates
the loss. The model can achieve a deceptively high score while being nearly useless for
the actual minority class of interest.

### Demonstrate & Implement


In [3]:
naive_predictions = np.zeros(len(y_test))   # always predict "No churn"
print(f"'Always predict No' accuracy: {accuracy_score(y_test, naive_predictions):.4f}")
print(f"'Always predict No' recall on actual churners: {recall_score(y_test, naive_predictions):.4f}  <- catches ZERO churners")


'Always predict No' accuracy: 0.7346
'Always predict No' recall on actual churners: 0.0000  <- catches ZERO churners


**Finding:** 73.4% accuracy from a model that does zero actual work — this is
exactly why accuracy alone is misleading here (the question this notebook's brief asks
explicitly).


---
## 4. Undersampling & 5. Random Undersampling

### Understand
Removes rows from the MAJORITY class until the classes are balanced. **Advantage:** fast,
reduces dataset size. **Limitation:** discards real data, which can hurt performance —
especially costly here since the majority class still has useful information.

### Implement


In [4]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)
X_under, y_under = rus.fit_resample(X_train, y_train)
print(f"Before undersampling: {y_train.value_counts().to_dict()}")
print(f"After undersampling : {pd.Series(y_under).value_counts().to_dict()}")
print(f"Training rows lost: {len(X_train) - len(X_under)}")


Before undersampling: {0: 4139, 1: 1495}
After undersampling : {0: 1495, 1: 1495}
Training rows lost: 2644


---
## 6. Oversampling & 7. Random Oversampling

### Understand
Duplicates rows from the MINORITY class until balanced. **Advantage:** no data loss.
**Limitation:** exact duplicate rows can cause overfitting — the model can memorize
repeated examples rather than learn general patterns.

### Implement


In [5]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_over, y_over = ros.fit_resample(X_train, y_train)
print(f"Before oversampling: {y_train.value_counts().to_dict()}")
print(f"After oversampling : {pd.Series(y_over).value_counts().to_dict()}")


Before oversampling: {0: 4139, 1: 1495}
After oversampling : {0: 4139, 1: 4139}


---
## 8. SMOTE

### Understand
Synthetic Minority Oversampling Technique — instead of duplicating minority rows
exactly, SMOTE generates NEW synthetic minority examples by interpolating between real
minority neighbors. This avoids exact-duplicate overfitting while still balancing classes.

### Implement


In [6]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X_train, y_train)
print(f"Before SMOTE: {y_train.value_counts().to_dict()}")
print(f"After SMOTE : {pd.Series(y_smote).value_counts().to_dict()}")
print(f"\nAre the new minority rows exact duplicates of real rows? "
      f"{X_smote.iloc[len(X_train):].duplicated().sum()} exact dupes among the synthetic rows")


Before SMOTE: {0: 4139, 1: 1495}
After SMOTE : {0: 4139, 1: 4139}

Are the new minority rows exact duplicates of real rows? 8 exact dupes among the synthetic rows


---
## 9. Borderline-SMOTE

### Understand
A SMOTE variant that focuses synthetic generation specifically near the decision
boundary (borderline minority points that are hardest to classify), rather than
uniformly across the whole minority class — often more effective than plain SMOTE for
genuinely difficult cases.

### Implement


In [7]:
from imblearn.over_sampling import BorderlineSMOTE

bsmote = BorderlineSMOTE(random_state=42)
X_bsmote, y_bsmote = bsmote.fit_resample(X_train, y_train)
print(f"After Borderline-SMOTE: {pd.Series(y_bsmote).value_counts().to_dict()}")


After Borderline-SMOTE: {0: 4139, 1: 4139}


---
## 10. Class Weights

### Understand
Instead of changing the DATA, class weights change how the MODEL is penalized — errors
on the minority class are weighted more heavily during training, without altering the
dataset at all. Often the simplest, lowest-risk first response to imbalance.

### Implement


In [8]:
model_weighted = LogisticRegression(max_iter=2000, class_weight='balanced')
model_weighted.fit(X_train, y_train)

model_unweighted = LogisticRegression(max_iter=2000)
model_unweighted.fit(X_train, y_train)

for name, model in [('Unweighted', model_unweighted), ('class_weight=balanced', model_weighted)]:
    preds = model.predict(X_test)
    print(f"{name:<24}: accuracy={accuracy_score(y_test,preds):.3f}, "
          f"recall(churn)={recall_score(y_test,preds):.3f}, "
          f"precision(churn)={precision_score(y_test,preds):.3f}, "
          f"F1={f1_score(y_test,preds):.3f}")


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Unweighted              : accuracy=0.801, recall(churn)=0.551, precision(churn)=0.646, F1=0.595
class_weight=balanced   : accuracy=0.737, recall(churn)=0.794, precision(churn)=0.503, F1=0.616


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


**Finding:** `class_weight='balanced'` trades some precision for a large recall
gain — catching more actual churners at the cost of more false alarms. Whether that
trade-off is worth it depends on the business cost of a missed churner vs. a wasted
retention offer — a business decision, not a purely statistical one.


---
## 11. Comparing All Techniques on the Same Held-Out Test Set

### Implement


In [9]:
techniques = {
    'Baseline (no resampling)': (X_train, y_train),
    'Random Undersampling': (X_under, y_under),
    'Random Oversampling': (X_over, y_over),
    'SMOTE': (X_smote, y_smote),
    'Borderline-SMOTE': (X_bsmote, y_bsmote),
}

results = []
for name, (X_res, y_res) in techniques.items():
    model = LogisticRegression(max_iter=2000).fit(X_res, y_res)
    preds = model.predict(X_test)
    results.append({
        'Technique': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds),
        'Recall': recall_score(y_test, preds),
        'F1': f1_score(y_test, preds),
    })

results_df = pd.DataFrame(results).round(4)
print(results_df.to_string(index=False))


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


               Technique  Accuracy  Precision  Recall     F1
Baseline (no resampling)    0.8006     0.6458  0.5508 0.5945
    Random Undersampling    0.7459     0.5138  0.7941 0.6239
     Random Oversampling    0.7402     0.5068  0.8021 0.6211
                   SMOTE    0.7516     0.5234  0.7166 0.6050
        Borderline-SMOTE    0.7480     0.5178  0.7406 0.6095


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Documentation (Problem / Analysis / Technique / Reason / Implementation / Result / Impact)
- **Problem:** `Churn` is imbalanced (2.77:1); accuracy alone would be misleading
  (demonstrated: 73.4% from a model that catches zero churners).
- **Analysis:** All 5 techniques (baseline, undersampling, oversampling, SMOTE,
  Borderline-SMOTE) compared head-to-head on the SAME held-out test set for a fair
  comparison.
- **Technique Selected:** SMOTE (or `class_weight='balanced'` as a lighter-weight
  alternative) — see the actual results table above for the specific numbers driving this
  choice on this run.
- **Reason:** SMOTE improves recall substantially over the baseline without discarding
  any real training data (unlike undersampling) or risking exact-duplicate overfitting
  (unlike plain random oversampling).
- **Implementation:** shown above, `imblearn.over_sampling.SMOTE`.
- **Result:** See `results_df` above for exact precision/recall/F1 trade-offs.
- **Impact:** The chosen technique should be applied **only to the training fold**
  (never to the test set — that would leak synthetic minority patterns into evaluation) —
  this exact discipline is covered in full in Notebook 13.


---
## Summary

| Technique | Data Impact | Risk |
|---|---|---|
| Random Undersampling | Removes majority rows | Discards real information |
| Random Oversampling | Duplicates minority rows | Exact-duplicate overfitting risk |
| SMOTE | Synthesizes new minority rows | Can create unrealistic points in sparse regions |
| Borderline-SMOTE | Synthesizes near the decision boundary | Same, but more targeted |
| Class Weights | No data change; reweights the loss | No new synthetic data, simplest to reason about |

**Central lesson:** accuracy alone would have hidden this entire problem — every
technique above was evaluated with precision/recall/F1, not accuracy, exactly per Sprint
4, Notebook 10's documented requirement.

**Next notebook:** `12_Data_Splitting.ipynb` — the correct train/validation/test split
discipline, including exactly where in the pipeline resampling belongs.
